In [1]:
# Client Setup
import boto3
import os
from dotenv import load_dotenv

load_dotenv(override=True)

region = os.environ.get("BEDROCK_REGION", "us-west-2")
client = boto3.client("bedrock-runtime", region_name=region)
model_id = os.environ["BEDROCK_MODEL_ID"]


In [2]:
# Helper functions


def add_user_message(messages, content):
    if isinstance(content, str):
        user_message = {"role": "user", "content": [{"text": content}]}
    else:
        user_message = {"role": "user", "content": content}
    messages.append(user_message)


def add_assistant_message(messages, content):
    if isinstance(content, str):
        assistant_message = {
            "role": "assistant",
            "content": [{"text": content}],
        }
    else:
        assistant_message = {"role": "assistant", "content": content}

    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    tool_choice="auto",
):
    params = {
        "modelId": model_id,
        "messages": messages,
        "inferenceConfig": {
            "temperature": temperature,
            "stopSequences": stop_sequences,
        },
    }

    if system:
        params["system"] = [{"text": system}]

    tool_choices = {
        "auto": {"auto": {}},
        "any": {"any": {}},
    }
    if tools:
        choice = tool_choices.get(tool_choice, {"tool": {"name": tool_choice}})
        params["toolConfig"] = {"tools": tools, "toolChoice": choice}

    response = client.converse(**params)
    parts = response["output"]["message"]["content"]

    return {
        "parts": parts,
        "stop_reason": response["stopReason"],
        "text": "\n".join([p["text"] for p in parts if "text" in p]),
    }

In [3]:
# Tool Schemas

article_details_schema = {
    "toolSpec": {
        "name": "article_details",
        "description": "This tool should be called with details about an article. It accepts information about the article's title, author, and related topics.",
        "inputSchema": {
            "json": {
                "type": "object",
                "properties": {
                    "title": {
                        "type": "string",
                        "description": "The title of the article. Can be left empty.",
                    },
                    "author": {
                        "type": "string",
                        "description": "The name of the article's author. Can be left empty.",
                    },
                    "topics": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "A list of topics or categories that the article covers. Can be an empty list.",
                    },
                },
            }
        },
    }
}

to_json_schema = {
    "toolSpec": {
        "name": "to_json",
        "description": "This tool processes any JSON data and can be used for generating structured content, transforming information, or creating any JSON-based output needed for your task.",
        "inputSchema": {
            "json": {"type": "object", "additionalProperties": True}
        },
    }
}

In [4]:
messages = []

add_user_message(
    messages,
    "Write a one-paragraph scholarly article about computer science. Include a title and author name",
)

result = chat(messages)

add_assistant_message(messages, result["text"])

result["text"]

"**Title:** Quantum-Classical Hybrid Algorithms: Bridging the Gap Between NISQ Devices and Practical Computational Supremacy\n\n**Author:** Dr. Sarah M. Chen, Department of Computer Science, Stanford University\n\nThe emergence of Noisy Intermediate-Scale Quantum (NISQ) devices has necessitated the development of hybrid quantum-classical algorithms that leverage the complementary strengths of both computational paradigms while mitigating the inherent limitations of current quantum hardware. This paper examines the theoretical foundations and practical implementations of variational quantum algorithms (VQAs), particularly focusing on the Quantum Approximate Optimization Algorithm (QAOA) and Variational Quantum Eigensolver (VQE), which employ classical optimization routines to iteratively refine quantum circuit parameters. Our analysis demonstrates that these hybrid approaches can achieve polynomial speedups for specific combinatorial optimization problems and quantum chemistry simulatio

In [5]:
messages = []

add_user_message(
    messages,
    f"""
Analyze the article below and extract key data. Then call the article_details tool.
                 
<article_text>
{result["text"]}                 
</article_text>
""",
)

json_result = chat(
    messages, tools=[article_details_schema], tool_choice="article_details"
)

json_result

{'parts': [{'toolUse': {'toolUseId': 'tooluse_bJQIdN4gjUL2Mis4R0QAei',
    'name': 'article_details',
    'input': {'title': 'Quantum-Classical Hybrid Algorithms: Bridging the Gap Between NISQ Devices and Practical Computational Supremacy',
     'author': 'Dr. Sarah M. Chen',
     'topics': ['quantum computing',
      'hybrid algorithms',
      'NISQ devices',
      'variational quantum algorithms',
      'QAOA',
      'VQE',
      'quantum optimization',
      'quantum chemistry',
      'quantum-classical interface',
      'error mitigation',
      'computational supremacy']},
    'type': 'tool_use'}}],
 'stop_reason': 'tool_use',
 'text': ''}

In [6]:
messages = []

add_user_message(
    messages,
    f"""
Analyze the article below and extract key data. Then call the to_json tool.
                 
<article_text>
{result["text"]}                 
</article_text>

When you call to_json, pass in the following structure:
{{
    "title": str # title of the article,
    "author": str # author of the article,
    "topics": List[str] # List of topics mentioned in the article,
    "num_topics: int # Number of topics mentioned
}}
""",
)

flexible_result = chat(messages, tools=[to_json_schema], tool_choice="to_json")

In [7]:
flexible_result

{'parts': [{'toolUse': {'toolUseId': 'tooluse_9eSnQyV3XZoIpAGAH4SyI4',
    'name': 'to_json',
    'input': {'title': 'Quantum-Classical Hybrid Algorithms: Bridging the Gap Between NISQ Devices and Practical Computational Supremacy',
     'author': 'Dr. Sarah M. Chen, Department of Computer Science, Stanford University',
     'topics': ['Quantum-Classical Hybrid Algorithms',
      'NISQ Devices',
      'Variational Quantum Algorithms',
      'Quantum Approximate Optimization Algorithm',
      'Variational Quantum Eigensolver',
      'Classical Optimization',
      'Combinatorial Optimization',
      'Quantum Chemistry Simulations',
      'Decoherence',
      'Gate Errors',
      'IBM Quantum',
      'Google Sycamore Processors',
      'Error Mitigation',
      'Quantum Advantage',
      'Fault-Tolerant Computing'],
     'num_topics': 15},
    'type': 'tool_use'}}],
 'stop_reason': 'tool_use',
 'text': ''}